# Four-model ink ensemble on the campaign-40 training segments

Segments: every training scroll of `campaign_archs_40.py` arm `holdout_n96_combined_surface_norm_nnpu` (`BASE_SCROLL_DICT`, 24 segments over 12 domains).

1. Setup — choose the third ensemble member with `SLOT3` (`"ink9-42"` = ink_9um seed 42, fast; `"v8in-1447"` = PHerc1447 fine-tune, slow) and the v8-in tile stride with `ve.V8IN_STRIDE`. Seed 43, dense_native and Hecate are always used.
2. Download + assemble the missing zarrs with `assemble_training_segments.py` (no norm / surface supervision recompute; existing masks are backed up first).
3. One cell per segment, then `render_all()` for an unattended run (skips finished segments, logs failures).

Saved (not displayed) to `output/test_visualizations/villa_ensemble/training_c40/`: `<name>_4models_d+0.png` (2x2 grid) and `<name>_ensemble_d+0.png` (ensemble over the composite view with a 1 cm bar), all at full resolution.

In [ ]:
import os
import gc
import re
import shutil
import subprocess
import sys
import time

if sys.executable.startswith("/opt/villa-venv"):
    raise RuntimeError("select the system Python 3.10 kernel (/usr/bin/python3); /opt/villa-venv is only used "
                       "internally for model inference")

REPO = "/vesuvius"
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import importlib
from utils import villa_ensemble as ve
ve = importlib.reload(ve)
import assemble_training_segments as ats

# third ensemble member: "ink9-42" (ink_9um hybrid_3d2d seed 42) or "v8in-1447" (PHerc1447 fine-tune, much slower)
SLOT3 = "v8in-1447"
ve.set_slot3(SLOT3)
# v8in tile stride: None = its default 21; 32 is 2.3x faster for a small fidelity loss (see utils/villa_ensemble.py)
ve.V8IN_STRIDE = 32

ZARR_DIR = ats.ZARR_DIR
OUT_DIR = os.path.join(REPO, "output", "test_visualizations", "villa_ensemble", "training_c40")
MASK_BACKUP_DIR = os.path.join(REPO, "backup_masks_training_c40")
ASSEMBLE_LOG = os.path.join(REPO, "_ves_tmp", "assemble_training_c40.log")
DEPTHS = (0,)
WRITE_DEPTH = 0
DIRECTION = "forward"
COMPILE = False
TAG = f"d{WRITE_DEPTH:+d}"
MIN_FREE_GB = 20

# campaign_archs_40.BASE_SCROLL_DICT (training scrolls of holdout_n96_combined_surface_norm_nnpu)
TRAIN_SCROLL_DICT = {
    "pherc0139": [20260115000000, 20260317000000, 20250223000000, 20250108000005, 20260112000000,
                  20260126000000, 20250831000000, 20260108000000, 20260302000000],
    "pherc0172": [20251111010954, 20251112000002],
    "pherc1667": [20240304141531, 20240304144031, 20231201215900],
    "pherc0009b": [20250919125754],
    "phercparis4": [20231210121321],
    "pherc0500p2": [20250628074500],
    "pherc0814": [20260226000000],
    "phercparis2_fr143": [20230301213755, 20230205142449],
    "pherc51cr4_fr8": [20231205222200],
    "phercparis1_fr34": [20230301213423],
    "pherc0343p": [20250511003658],
    "pherc0841": [20260221022814],
}
# rendered after the campaign segments; not part of BASE_SCROLL_DICT
EXTRA_SCROLL_DICT = {"pherc0343": [20261007000000], "pherc1447": [20260930144758, 20260930144760, 20260930144762]}
SEG_NAME = {zid: name for name, _seg, zid in ats.SEGMENTS}
FRAGMENTS = {SEG_NAME[str(zid)]: str(zid) for ids in (*TRAIN_SCROLL_DICT.values(), *EXTRA_SCROLL_DICT.values())
             for zid in ids}
DOMAIN = {str(zid): domain for scrolls in (TRAIN_SCROLL_DICT, EXTRA_SCROLL_DICT) for domain, ids in scrolls.items()
          for zid in ids}
# render pixel size: the 9.362 µm training grid, except level-2 pooled fine scans and the 9.366 µm p841 crop
ve.EXTRA_NATIVE_UM.update({zid: 9.362 for zid in FRAGMENTS.values()})
ve.EXTRA_NATIVE_UM.update({"20240304141531": 9.596, "20240304144031": 9.596, "20231210121321": 9.6,
                           "20260221022814": 9.366})
TITLES = {"dnative": "dense_native (16k)", "hecate": "Hecate 9.6µm", "ink9-42": "ink_9um seed 42",
          "v8in-1447": "v8-in PHerc1447 LOO-w062", "ink9-43": "ink_9um seed 43"}


def is_assembled(zid):
    zpath = os.path.join(ZARR_DIR, f"{zid}.zarr")
    return os.path.isfile(os.path.join(zpath, ".zarray")) and os.path.isfile(os.path.join(REPO, "masks", f"{zid}.png"))


def outputs(name):
    return [os.path.join(OUT_DIR, f"{name}_{kind}_{TAG}.png") for kind in ("4models", "ensemble")]


print(f"models: {[short for short, _ in ve.MODELS]} | {len(FRAGMENTS)} segments | zarrs: {ZARR_DIR} | out: {OUT_DIR}")

In [ ]:
# --- download + assemble the missing training zarrs (resumable: finished segments are skipped) ---
from tqdm.auto import tqdm

todo = [name for name, zid in FRAGMENTS.items() if not is_assembled(zid)]
print(f"{len(FRAGMENTS) - len(todo)} assembled, {len(todo)} to do: {todo}")
print(f"free disk: {shutil.disk_usage(REPO).free / 1e9:.0f} GB")
if todo:
    if shutil.disk_usage(REPO).free / 1e9 < MIN_FREE_GB:
        raise RuntimeError(f"less than {MIN_FREE_GB} GB free")
    # the assembler rewrites masks/<id>.png; keep the current ones
    os.makedirs(MASK_BACKUP_DIR, exist_ok=True)
    for name in todo:
        source = os.path.join(REPO, "masks", f"{FRAGMENTS[name]}.png")
        backup = os.path.join(MASK_BACKUP_DIR, f"{FRAGMENTS[name]}.png")
        if os.path.isfile(source) and not os.path.isfile(backup):
            shutil.copy2(source, backup)
    command = [sys.executable, "-u", "assemble_training_segments.py", "--only", ",".join(todo),
               "--skip-norm", "--skip-surface", "--concurrent-fragments", "3"]
    shown = re.compile(r"\[done\]|\[FAIL\]|\[1/3\]|wrote|Error|WARN|integrity|SUMMARY", re.I)
    bar = tqdm(total=len(todo), desc="assemble", unit="segment")
    with open(ASSEMBLE_LOG, "a", encoding="utf-8") as log, subprocess.Popen(
            command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            log.write(line)
            if line.startswith(("[done]", "[FAIL]")):
                bar.update(1)
            if shown.search(line):
                bar.write(line.rstrip())
        proc.wait()
    bar.close()
    print(f"assembler exit {proc.returncode}; full log: {ASSEMBLE_LOG}")

for name, zid in FRAGMENTS.items():
    ok, msg = ats._zarr_integrity(zid) if is_assembled(zid) else (False, "MISSING")
    print(f"  {'OK  ' if ok else 'FAIL'} {name:<14} {zid}  {msg}")
print(f"free disk: {shutil.disk_usage(REPO).free / 1e9:.0f} GB")

In [ ]:
def render(name, force=False):
    scroll_id = FRAGMENTS[name]
    grid_path, ens_path = outputs(name)
    if not force and os.path.isfile(grid_path) and os.path.isfile(ens_path):
        print(f"[skip] {name}: already rendered")
        return
    if not is_assembled(scroll_id):
        raise RuntimeError(f"{name} ({scroll_id}) is not assembled; run the assembly cell")
    t0 = time.time()
    ve.ensure_villa_input(scroll_id, ZARR_DIR)
    ve.run_models(scroll_id, DEPTHS, [DIRECTION], compile_model=COMPILE)
    print("[ensemble] stretching + z-score ensemble ...", flush=True)
    stretched, ensemble, stats = ve.ensemble(scroll_id, DEPTHS, DIRECTION, WRITE_DEPTH)
    label = f"{DOMAIN[scroll_id]} {name}"
    ve.save_grid([(f"{label}  {TITLES[k]}  {TAG} {DIRECTION}", stretched[k]) for k, _ in ve.MODELS], 2, grid_path)
    ve.save_ensemble_figure(scroll_id, ensemble, f"{label}  4-model ensemble (mean z-score)  {TAG} {DIRECTION}",
                            ens_path)
    del stretched, ensemble, stats
    gc.collect()
    print(f"[saved] {grid_path}\n[saved] {ens_path}\n[done] {name} in {time.time() - t0:.0f}s")


def render_all(names=None):
    """unattended: render every segment, logging failures instead of stopping; finished maps/figures are cached"""
    failed = []
    for name in names or FRAGMENTS:
        try:
            render(name)
        except Exception as exc:
            failed.append(name)
            print(f"[FAILED] {name}: {exc}", flush=True)
    print(f"[render_all] {len(failed)} failed: {failed}")

In [ ]:
# --- PHerc0814 seg46527 (sanity: known ink) ---
render("seg46527")

In [ ]:
# --- PHerc0139 w044 ---
render("w044")

In [ ]:
# --- PHerc0139 w035 ---
render("w035")

In [ ]:
# --- PHerc0139 w059 ---
render("w059")

In [ ]:
# --- PHerc0139 w030 ---
render("w030")

In [ ]:
# --- PHerc0139 w043 ---
render("w043")

In [ ]:
# --- PHerc0139 w045 ---
render("w045")

In [ ]:
# --- PHerc0139 w040 ---
render("w040")

In [ ]:
# --- PHerc0139 w041 ---
render("w041")

In [ ]:
# --- PHerc0139 w039 ---
render("w039")

In [ ]:
# --- PHerc0172 w068 (7.91 µm, resampled) ---
render("w068")

In [ ]:
# --- PHerc0172 w087 (7.91 µm, resampled) ---
render("w087")

In [ ]:
# --- PHerc1667 w013 (2.4 µm level 2, depth-pooled) ---
render("w013")

In [ ]:
# --- PHerc1667 w018 (2.4 µm level 2, depth-pooled; large) ---
render("w018")

In [ ]:
# --- PHerc1667 Cr1Fr3 (3.24 µm, resampled) ---
render("cr1fr3")

In [ ]:
# --- PHerc0009B auto_grown 487 (8.64 µm, resampled) ---
render("p9b_487")

In [ ]:
# --- PHercParis4 (2.4 µm level 2, depth-pooled) ---
render("paris4")

In [ ]:
# --- PHerc0500P2 front ---
render("500P2_front")

In [ ]:
# --- PHercParis2 Fr143 (88 keV rescan) ---
render("paris2_fr143")

In [ ]:
# --- PHercParis2 Fr47 (88 keV rescan) ---
render("paris2_fr47")

In [ ]:
# --- PHerc51 Cr4 Fr8 (88 keV rescan) ---
render("scroll6_fr8")

In [ ]:
# --- PHercParis1 Fr34 (88 keV rescan) ---
render("paris1_fr34")

In [ ]:
# --- PHerc0343P (8.64 µm, resampled) ---
render("p343")

In [ ]:
# --- PHerc0841 auto_grown (cropped) ---
render("p841")

In [ ]:
# --- PHerc0343 First Letters segment (extra, not in campaign 40) ---
render("p343_letters")


In [ ]:
# --- PHerc1447 windings (extra; v8-in was fine-tuned on w058/w060 with w062 left out) ---
for name in ("ph1447_w058", "ph1447_w060", "ph1447_w062"):
    render(name)


In [ ]:
# --- unattended: everything not yet rendered ---
render_all()